# Task 2 – Cleaning
**Who:** Joud & Maram (together)  
**Input:** `Food_Inspections.csv` (raw data from the Chicago Data Portal, in this project folder)  
**Output:** `food_clean.csv`, the clean file both feature notebooks start from.

The cleaning rules below are our **suggested** rules. If we change a rule after discussing, we change the cell and the note under it.
After each rule we print the number of rows, so we can see how much data each rule removes.

In [33]:
import pandas as pd

We only need pandas in this notebook (no plots).

In [34]:
food = pd.read_csv('Food_Inspections.csv')
food['Inspection Date'] = pd.to_datetime(food['Inspection Date'])
print(food.shape)

(317175, 17)


We read the raw file and turn the date text into a real date, so we can compare dates later.
**Rows at the start:** 317,176 rows and 17 columns.

In [35]:
# Rule 1: keep only real inspection results
df = food[food['Results'].isin(['Pass', 'Pass w/ Conditions', 'Fail'])].copy()
print(df.shape)

(272286, 17)


**Rule 1.** Out of Business, No Entry, Not Ready and Business Not Located are not real inspections (nobody checked the food), so we remove them. We keep only Pass, Pass w/ Conditions and Fail.

In [36]:
# Rule 2: remove exact duplicate rows
df = df.drop_duplicates()
print(df.shape)

(272286, 17)


**Rule 2.** The dataset description warns that some rows are duplicated. Two identical rows are one inspection, so we keep one.

In [37]:
# Rule 3: remove the same business inspected twice on the same date
df = df.drop_duplicates(subset=['License #', 'Inspection Date'])
print(df.shape)

(271222, 17)


**Rule 3.** If one business (same License #) has two rows on the same day, it is the same visit recorded twice. We keep the first one.

In [38]:
# Rule 4: remove rows without a License #
df = df.dropna(subset=['License #'])
df = df[df['License #'] != 0]
print(df.shape)

(270583, 17)


**Rule 4.** We need the License # to follow each business over time (previous result, days since last inspection). Rows with no License # or License # = 0 cannot be followed, so we remove them.

In [39]:
# Rule 5: fill missing text values
df['Facility Type'] = df['Facility Type'].fillna('Unknown')
df['Risk'] = df['Risk'].fillna('Unknown')
df['Risk'] = df['Risk'].replace('All', 'Unknown')
df['Risk'].value_counts()

Risk
Risk 1 (High)      206211
Risk 2 (Medium)     46807
Risk 3 (Low)        17546
Unknown                19
Name: count, dtype: int64

**Rule 5.** Facility Type and Risk are text columns with a few missing values. Instead of deleting those rows we write "Unknown". Risk = "All" is not a real risk level, so it also becomes "Unknown".

In [40]:
# Rule 6: one ID per business
names_per_license = df.groupby('License #')['DBA Name'].nunique()
print((names_per_license > 1).sum(), 'licenses have more than one business name')
print(df['License #'].nunique(), 'licenses in total')

1134 licenses have more than one business name
43891 licenses in total


**Rule 6 (check).** Some license numbers are used by businesses with different names over time (for example License # 980: "7400 S HALSTED FOOD AND LIQUORS" in 2015–2017, then "SPIRITS BEVERAGE CENTER" in 2018–2020). If we followed a business by License # only, the history of one business would be mixed with another. **Result:** about 70 licenses have more than one business name, a very small share of all licenses (write the total from the output here.)

In [41]:
df['Business_ID'] = df['License #'].astype(int).astype(str) + '_' + df['DBA Name'].str.upper().str.strip()
df['Business_ID'].nunique()

45105

**Rule 6.** We define one business as **License # + business name** (`Business_ID`). The name is changed to upper case and the spaces at the start and end are removed (`upper()`, `strip()`), so small writing differences do not create two businesses. `astype(int)` removes the ".0" from the license number (it was read as a decimal number), so the ID looks like `1000_BELMONTE LIQUORS`. All history columns (previous result, days since last inspection, past fail rate…) use `Business_ID` instead of `License #`.

In [42]:
risks_per_business = df.groupby('Business_ID')['Risk'].nunique()
print((risks_per_business > 1).sum(), 'businesses changed risk level')

45 businesses changed risk level


Risk can also change over time for the same business (for example when it starts to cook hot food). Risk is recorded **per inspection**, so we keep the value recorded at each inspection and do not give one fixed risk to each business. (Write here the number.)

In [43]:
df['First_Seen_Date'] = df.groupby('Business_ID')['Inspection Date'].transform('min')
df[['Business_ID', 'Inspection Date', 'First_Seen_Date']].head()

,Business_ID,Inspection Date,First_Seen_Date
0,3104395_LA MICHOACANA,2026-10-16,2026-10-16
1,2993280_IRON CAFE,2026-10-16,2026-10-16
2,22951_DEWEY,2026-10-16,2010-06-09
4,3016043_STEPPING STONES NURSERY SCHOOL,2026-10-16,2025-04-01
5,2659225_MAXI BAKERY,2026-10-16,2019-04-18


**First time we see each business.** `transform('min')` gives every row the earliest inspection date of its business. We do this **before** keeping only 2015–2022, so the history goes back to 2010 (the start of the dataset). Maram's notebook uses it to calculate how long the business has been inspected (an approximation of the business's age; businesses older than 2010 will look as if they started in 2010).

In [44]:
# Rule 7: keep the years we study
df['Year'] = df['Inspection Date'].dt.year
df = df[(df['Year'] >= 2015) & (df['Year'] <= 2022)]
print(df.shape)

(126787, 20)


**Rule 7.** We keep 2015–2022: about 3.5 years before the July 2018 change and 3.5 years after the transition year. We add 2021–2022 because 2020 had COVID (fewer inspections), so 2020 alone is not a normal "after" year.

In [45]:
# Target column: what the model will predict
def make_target(result):
    if result == 'Fail':
        return 1
    else:
        return 0

df['Fail'] = df['Results'].apply(make_target)
df['Fail'].value_counts()

Fail
0    98432
1    28355
Name: count, dtype: int64

**Target.** Fail = 1, Pass and Pass w/ Conditions = 0. This is the column the model will predict.

In [46]:
# Period column (same def as Task 1, with the 90-day grace period)
def get_period(date):
    if date < pd.Timestamp('2018-07-01'):
        return 'Pre'
    elif date < pd.Timestamp('2018-10-01'):
        return 'Grace (90 days)'
    elif date < pd.Timestamp('2019-07-01'):
        return 'Late transition'
    else:
        return 'Post'

df['Period'] = df['Inspection Date'].apply(get_period)
df['Period'].value_counts()

Period
Pre                64293
Post               47780
Late transition    11352
Grace (90 days)     3362
Name: count, dtype: int64

**Period.** Before the change, the 90-day grace period, the rest of the transition year, and after. We add it here so both feature notebooks already have it.

In [47]:
df.isnull().sum()

Inspection ID          0
DBA Name               0
AKA Name             327
License #              0
Facility Type          0
Risk                   0
Address                0
City                  46
State                 34
Zip                    1
Inspection Date        0
Inspection Type        1
Results                0
Violations         17814
Latitude             455
Longitude            455
Location             455
Business_ID            0
First_Seen_Date        0
Year                   0
Fail                   0
Period                 0
dtype: int64

Check what is still missing. **Result:** only columns we do not use for the analysis still have a few missing values (AKA Name, City, State, Zip, Latitude, Longitude, Location and one Inspection Type), so we leave them. Violations is empty when the inspector found nothing, which is normal (it means 0 violations).

In [48]:
df.to_csv('food_clean.csv', index=False)
print(df.shape)

(126787, 22)


We save the clean file. **Rows at the end:** 126,787 rows (2015–2022).. `food_clean.csv` is not uploaded to GitHub (see `.gitignore`), so each of us runs this notebook once on our own computer.